# TP Olist : découverte, audit et nettoyage des données

Dans ce notebook, j'essaie de comprendre les données Olist (un site de vente en ligne brésilien), de repérer ce qui ne va pas, puis de les nettoyer.

Mon plan :
1. **Charger** les 9 fichiers CSV
2. **Comprendre** ce qu'il y a dedans et comment les fichiers sont reliés
3. **Faire l'audit** : je compte les problèmes, mais je ne modifie encore rien
4. **Nettoyer** les problèmes trouvés
5. **Regrouper** les tables qui ont plusieurs lignes par commande
6. **Assembler** une table finale avec une ligne par commande

Mes explications plus détaillées sont dans `CARNET.md`.

# 0. Chargement des fichiers

Je lis les 9 CSV avec `pd.read_csv`. Pour les colonnes qui contiennent des dates, j'utilise `parse_dates` : sinon pandas les lit comme du texte et je ne peux pas les comparer entre elles.

In [ ]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 50)   # pour voir toutes les colonnes
DATA = Path("/workspace/donnees/olist")    # dossier où sont les CSV

def lire(nom, dates=None):
    return pd.read_csv(DATA / nom, parse_dates=dates)

customers   = lire("olist_customers_dataset.csv")
geo         = lire("olist_geolocation_dataset.csv")
items       = lire("olist_order_items_dataset.csv", ["shipping_limit_date"])
payments    = lire("olist_order_payments_dataset.csv")
reviews     = lire("olist_order_reviews_dataset.csv", ["review_creation_date", "review_answer_timestamp"])
orders      = lire("olist_orders_dataset.csv", ["order_purchase_timestamp", "order_approved_at",
                                                "order_delivered_carrier_date", "order_delivered_customer_date",
                                                "order_estimated_delivery_date"])
products    = lire("olist_products_dataset.csv")
sellers     = lire("olist_sellers_dataset.csv")
translation = lire("product_category_name_translation.csv")

# Je range tout dans un dictionnaire pour pouvoir faire des boucles sur les 9 tables
tables = {"customers": customers, "geolocation": geo, "order_items": items, "order_payments": payments,
          "order_reviews": reviews, "orders": orders, "products": products,
          "sellers": sellers, "translation": translation}

# 1. Comprendre les données

## 1.1 Taille et colonnes de chaque fichier

In [ ]:
pd.DataFrame({nom: {"lignes": len(df), "colonnes": df.shape[1],
                    "liste des colonnes": ", ".join(df.columns)}
              for nom, df in tables.items()}).T

Je regarde à quoi ressemble la table principale, `orders` :

In [ ]:
orders.head(3)

## 1.2 Comment les tables sont reliées

En lisant les noms de colonnes, je vois que certaines reviennent dans plusieurs fichiers (`order_id`, `customer_id`, `product_id`…). Ce sont les **clés** qui relient les tables. J'ai fait ce schéma :

```
customers ──customer_id── orders ──order_id── order_items ──product_id── products ──category── translation
    │                        │                     └──seller_id── sellers
    └─zip_code_prefix─┐      ├──order_id── order_payments
                      │      └──order_id── order_reviews
                  geolocation (zip_code_prefix)
```

La table au centre, c'est `orders` (les commandes).

Ensuite je vérifie, pour chaque clé, s'il y a **une seule ligne** par valeur ou **plusieurs**. Par exemple : est-ce qu'une commande apparaît une seule fois dans `order_items`, ou plusieurs fois ?

In [ ]:
def verifier_cle(nom, df, col):
    return {"table": nom, "clé": col,
            "valeurs différentes": df[col].nunique(), "lignes": len(df),
            "résultat": "une ligne par clé" if df[col].is_unique else "plusieurs lignes par clé"}

pd.DataFrame([
    verifier_cle("orders", orders, "order_id"),
    verifier_cle("orders", orders, "customer_id"),
    verifier_cle("customers", customers, "customer_id"),
    verifier_cle("customers", customers, "customer_unique_id"),
    verifier_cle("order_items", items, "order_id"),
    verifier_cle("order_payments", payments, "order_id"),
    verifier_cle("order_reviews", reviews, "order_id"),
    verifier_cle("products", products, "product_id"),
    verifier_cle("sellers", sellers, "seller_id"),
    verifier_cle("geolocation", geo, "geolocation_zip_code_prefix"),
])

Ce que j'en retiens :
- Il y a autant de `customer_id` que de commandes. En fait, un nouveau `customer_id` est créé **à chaque commande**. Pour retrouver une même personne qui commande plusieurs fois, il faut utiliser `customer_unique_id`.
- `order_items`, `order_payments`, `order_reviews` et `geolocation` ont **plusieurs lignes pour une même clé**. Si je les fusionne directement avec `orders`, des commandes vont se retrouver en double. Il faudra d'abord les regrouper (partie 4).

# 2. Audit de qualité

L'idée de cette partie : **avant de nettoyer, je mesure**. Je compte les problèmes, mais je ne touche pas encore aux données.

Je cherche 5 types de problèmes :
1. les valeurs manquantes
2. les doublons
3. les dates incohérentes
4. les incohérences entre fichiers
5. les valeurs aberrantes

Pour ne rien oublier, je note chaque problème trouvé dans une liste `audit` avec une petite fonction `note()`. À la fin (2.6), j'affiche tout dans un seul tableau.

In [ ]:
audit = []

def note(fichier, famille, probleme, n, total):
    """Ajoute un problème à la liste et renvoie le nombre de lignes concernées."""
    audit.append({"fichier": fichier, "famille": famille, "problème": probleme,
                  "lignes": int(n), "%": round(100 * n / total, 2)})
    return int(n)

## 2.1 Valeurs manquantes

J'utilise `isna().sum()` qui compte les cases vides dans chaque colonne.

In [ ]:
for nom, df in tables.items():
    for col, n in df.isna().sum().items():
        if n > 0:
            note(nom, "Valeurs manquantes", f"{col} vide", n, len(df))

pd.DataFrame([a for a in audit if a["famille"] == "Valeurs manquantes"])

Dans `orders`, il manque des dates de livraison. Je me demande si c'est normal : une commande annulée ou pas encore livrée n'a logiquement pas de date de livraison. Je vérifie en croisant avec le statut :

In [ ]:
pd.crosstab(orders.order_status, orders.order_delivered_customer_date.isna(),
            colnames=["date de livraison vide ?"])

## 2.2 Doublons

Je cherche deux sortes de doublons :
- les lignes **complètement identiques** (avec `duplicated()`),
- les **identifiants qui devraient être uniques** mais qui apparaissent plusieurs fois.

In [ ]:
# Lignes identiques dans chaque table
for nom, df in tables.items():
    n = df.duplicated().sum()
    if n > 0:
        note(nom, "Doublons", "lignes complètement identiques", n, len(df))

# Identifiants répétés
note("order_reviews", "Doublons", "review_id qui apparaît plusieurs fois", reviews.review_id.duplicated().sum(), len(reviews))
note("order_reviews", "Doublons", "commande avec plusieurs avis", reviews.order_id.duplicated().sum(), len(reviews))
note("geolocation", "Doublons", "code postal avec plusieurs coordonnées GPS",
     geo.drop_duplicates().geolocation_zip_code_prefix.duplicated().sum(), len(geo))

pd.DataFrame([a for a in audit if a["famille"] == "Doublons"])

## 2.3 Cohérence des dates

Une commande suit normalement cet ordre :

**achat → validation du paiement → remise au transporteur → livraison au client**

Donc une date ne devrait jamais être avant la précédente. Je compte les commandes où ce n'est pas le cas.

Remarque : quand une des deux dates est vide, la comparaison donne `False`, donc ces lignes ne sont pas comptées comme des erreurs.

In [ ]:
o = orders   # nom plus court

regles = {
    "validation avant l'achat":         o.order_approved_at < o.order_purchase_timestamp,
    "transporteur avant la validation": o.order_delivered_carrier_date < o.order_approved_at,
    "transporteur avant l'achat":       o.order_delivered_carrier_date < o.order_purchase_timestamp,
    "livraison avant le transporteur":  o.order_delivered_customer_date < o.order_delivered_carrier_date,
    "livraison avant l'achat":          o.order_delivered_customer_date < o.order_purchase_timestamp,
    "livraison prévue avant l'achat":   o.order_estimated_delivery_date < o.order_purchase_timestamp,
}
for probleme, erreurs in regles.items():
    note("orders", "Cohérence temporelle", probleme, erreurs.sum(), len(o))

# Pour les avis : on ne peut pas répondre à un avis avant qu'il soit créé
note("order_reviews", "Cohérence temporelle", "réponse avant la création de l'avis",
     (reviews.review_answer_timestamp < reviews.review_creation_date).sum(), len(reviews))

pd.DataFrame([a for a in audit if a["famille"] == "Cohérence temporelle"])

Je regarde aussi sur quelle période vont les données, et combien il y a de commandes par mois. S'il y a très peu de commandes au début ou à la fin, c'est sûrement que ces mois ne sont pas complets.

In [ ]:
print("Première commande :", o.order_purchase_timestamp.min())
print("Dernière commande :", o.order_purchase_timestamp.max())
o.order_purchase_timestamp.dt.to_period("M").value_counts().sort_index()

## 2.4 Cohérence entre les fichiers

Si un `order_id` est dans `order_items`, il devrait aussi exister dans `orders`. Sinon, l'article appartient à une commande « fantôme ». J'appelle ça des **orphelins**.

Pour les trouver, j'utilise `isin()` : il me dit si chaque valeur existe dans l'autre table.

In [ ]:
def orphelins(fichier, probleme, enfant, col_enfant, parent, col_parent):
    absents = ~enfant[col_enfant].isin(parent[col_parent])   # le ~ veut dire "pas"
    return note(fichier, "Cohérence entre fichiers", probleme, absents.sum(), len(enfant))

orphelins("order_items",    "order_id inconnu dans orders",          items, "order_id", orders, "order_id")
orphelins("order_payments", "order_id inconnu dans orders",          payments, "order_id", orders, "order_id")
orphelins("order_reviews",  "order_id inconnu dans orders",          reviews, "order_id", orders, "order_id")
orphelins("orders",         "customer_id inconnu dans customers",    orders, "customer_id", customers, "customer_id")
orphelins("order_items",    "product_id inconnu dans products",      items, "product_id", products, "product_id")
orphelins("order_items",    "seller_id inconnu dans sellers",        items, "seller_id", sellers, "seller_id")
orphelins("customers",      "code postal inconnu dans geolocation",  customers, "customer_zip_code_prefix", geo, "geolocation_zip_code_prefix")
orphelins("sellers",        "code postal inconnu dans geolocation",  sellers, "seller_zip_code_prefix", geo, "geolocation_zip_code_prefix")

# Dans l'autre sens : des commandes sans rien derrière
orphelins("orders", "commande sans aucun article", orders, "order_id", items, "order_id")
orphelins("orders", "commande sans paiement",      orders, "order_id", payments, "order_id")
orphelins("orders", "commande sans avis",          orders, "order_id", reviews, "order_id")

# Catégories de produits qui n'ont pas de traduction
categories = products.product_category_name.dropna()
note("products", "Cohérence entre fichiers", "catégorie sans traduction",
     (~categories.isin(translation.product_category_name)).sum(), len(products))
print("Catégories sans traduction :", set(categories) - set(translation.product_category_name))

Deux autres vérifications :
- est-ce que le **statut** de la commande correspond à ses dates ? (une commande « livrée » doit avoir une date de livraison)
- est-ce que le **montant payé** correspond bien au prix des articles + les frais de port ?

In [ ]:
note("orders", "Cohérence entre fichiers", "statut 'delivered' mais pas de date de livraison",
     ((o.order_status == "delivered") & o.order_delivered_customer_date.isna()).sum(), len(o))
note("orders", "Cohérence entre fichiers", "statut 'canceled' mais une date de livraison",
     ((o.order_status == "canceled") & o.order_delivered_customer_date.notna()).sum(), len(o))

# Total des articles (prix + port) par commande, comparé au total payé
total_articles = items.groupby("order_id")[["price", "freight_value"]].sum().sum(axis=1)
total_paye     = payments.groupby("order_id").payment_value.sum()
ecart = (total_paye - total_articles).dropna().abs()

note("order_payments", "Cohérence entre fichiers", "montant payé différent de prix + port (écart > 1 R$)",
     (ecart > 1).sum(), len(ecart))
ecart[ecart > 1].describe()

## 2.5 Valeurs aberrantes

Une valeur aberrante, c'est une valeur impossible (un prix négatif, un poids de 0 g…) ou très éloignée des autres.

Je commence par `describe()`, qui donne le minimum, le maximum, la moyenne et les quartiles de chaque colonne. Ça permet de repérer rapidement les valeurs bizarres.

In [ ]:
items[["price", "freight_value"]].describe()

In [ ]:
payments[["payment_value", "payment_installments"]].describe()

In [ ]:
products.select_dtypes("number").describe()

Pour les colonnes de texte, j'utilise `value_counts()` pour voir toutes les valeurs possibles :

In [ ]:
payments.payment_type.value_counts()

Je compte maintenant les valeurs qui me semblent impossibles :

In [ ]:
note("order_items",    "Valeurs aberrantes", "prix ≤ 0",                         (items.price <= 0).sum(), len(items))
note("order_items",    "Valeurs aberrantes", "frais de port = 0",                (items.freight_value == 0).sum(), len(items))
note("order_payments", "Valeurs aberrantes", "paiement de 0",                    (payments.payment_value == 0).sum(), len(payments))
note("order_payments", "Valeurs aberrantes", "paiement en 0 fois",               (payments.payment_installments == 0).sum(), len(payments))
note("order_payments", "Valeurs aberrantes", "type de paiement 'not_defined'",   (payments.payment_type == "not_defined").sum(), len(payments))
note("products",       "Valeurs aberrantes", "poids de 0 g",                     (products.product_weight_g == 0).sum(), len(products))

# Le Brésil est à peu près entre -34 et 6 de latitude, et entre -74 et -34 de longitude
hors_bresil = ~geo.geolocation_lat.between(-34, 6) | ~geo.geolocation_lng.between(-74, -34)
note("geolocation", "Valeurs aberrantes", "coordonnées GPS en dehors du Brésil", hors_bresil.sum(), len(geo))

Pour les valeurs **très grandes**, j'utilise la méthode de l'écart interquartile (IQR) vue en stats :
- Q1 = la valeur sous laquelle se trouvent 25 % des données, Q3 = celle sous laquelle se trouvent 75 %
- IQR = Q3 − Q1
- une valeur est considérée comme extrême si elle dépasse **Q3 + 1,5 × IQR**

Attention : extrême ne veut pas dire faux. Un frigo coûte plus cher qu'un stylo, c'est normal.

In [ ]:
def valeurs_extremes(fichier, df, col):
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    seuil = q3 + 1.5 * (q3 - q1)
    n = note(fichier, "Valeurs aberrantes", f"{col} > {seuil:.0f} (méthode IQR)", (df[col] > seuil).sum(), len(df))
    print(f"{col} : {n} valeurs au-dessus de {seuil:.1f} (le max est {df[col].max()})")

valeurs_extremes("order_items", items, "price")
valeurs_extremes("order_items", items, "freight_value")
valeurs_extremes("order_payments", payments, "payment_value")
valeurs_extremes("products", products, "product_weight_g")

Dernière chose : les noms de villes. J'ai l'impression qu'une même ville peut être écrite de plusieurs façons (avec ou sans accent, en majuscules…). Je compare le nombre de villes avant et après avoir tout mis en minuscules sans accents :

In [ ]:
villes = geo.geolocation_city
villes_simplifiees = (villes.str.lower()
                      .str.normalize("NFKD").str.encode("ascii", "ignore").str.decode("ascii"))
print("Nombre de villes différentes           :", villes.nunique())
print("Après minuscules et sans les accents  :", villes_simplifiees.nunique())

## 2.6 Bilan de l'audit

Voici tous les problèmes trouvés, rangés par fichier. Je m'en sers pour décider quoi nettoyer et pour remplir `CARNET.md`.

In [ ]:
bilan = (pd.DataFrame(audit)
         .query("lignes > 0")                      # je garde seulement les vrais problèmes
         .sort_values(["fichier", "famille", "lignes"], ascending=[True, True, False])
         .reset_index(drop=True))
bilan

Le même bilan résumé : combien de problèmes de chaque type par fichier.

In [ ]:
bilan.pivot_table(index="fichier", columns="famille", values="lignes", aggfunc="count", fill_value=0)

# 3. Nettoyage

Maintenant que je sais ce qui ne va pas, je corrige. Je travaille sur des **copies** (`.copy()`) pour garder les données d'origine intactes, au cas où je me tromperais.

In [ ]:
orders_propre   = orders.copy()
products_propre = products.copy()
geo_propre      = geo.copy()
reviews_propre  = reviews.copy()

## 3.1 orders : calculer les délais

À partir des dates, je calcule deux nouvelles colonnes utiles pour l'analyse :
- le **délai de livraison** en jours (entre l'achat et la livraison)
- le **retard** en jours (entre la date prévue et la date réelle ; s'il est positif, la commande est en retard)

In [ ]:
orders_propre["delai_livraison_jours"] = (orders_propre.order_delivered_customer_date
                                          - orders_propre.order_purchase_timestamp).dt.days
orders_propre["retard_jours"] = (orders_propre.order_delivered_customer_date
                                 - orders_propre.order_estimated_delivery_date).dt.days
orders_propre["en_retard"] = orders_propre["retard_jours"] > 0

orders_propre[["delai_livraison_jours", "retard_jours"]].describe()

## 3.2 products : colonnes, catégories vides et traduction

- Deux colonnes ont une faute d'orthographe (`lenght` au lieu de `length`), je les renomme.
- 610 produits n'ont pas de catégorie : je ne veux pas les supprimer car ils ont été vendus, donc je mets `"inconnue"`.
- J'ajoute le nom de la catégorie en anglais grâce à la table `translation`.

In [ ]:
products_propre = products_propre.rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length",
})
products_propre["product_category_name"] = products_propre["product_category_name"].fillna("inconnue")

products_propre = products_propre.merge(translation, on="product_category_name", how="left")
# Si pas de traduction, je garde le nom portugais
products_propre["product_category_name_english"] = (products_propre["product_category_name_english"]
                                                    .fillna(products_propre["product_category_name"]))
products_propre.isna().sum()

## 3.3 geolocation : doublons et coordonnées fausses

Je supprime les lignes identiques, puis les points qui ne sont pas au Brésil.

In [ ]:
print("Au départ                   :", len(geo_propre))
geo_propre = geo_propre.drop_duplicates()
print("Sans les doublons            :", len(geo_propre))
geo_propre = geo_propre[geo_propre.geolocation_lat.between(-34, 6) & geo_propre.geolocation_lng.between(-74, -34)]
print("Sans les points hors Brésil  :", len(geo_propre))

## 3.4 reviews : les commandes avec plusieurs avis

Je regarde à quoi ressemblent ces cas avant de décider quoi faire :

In [ ]:
reviews_propre[reviews_propre.order_id.duplicated(keep=False)].sort_values("order_id").head(6)

# 4. Regrouper (agrégations)

Problème : certaines tables ont **plusieurs lignes pour une seule commande** (plusieurs articles, plusieurs paiements…). Je ne veux pas supprimer ces lignes car je perdrais de l'information.

Solution : je les **résume** avec `groupby()` + `agg()`. Par exemple, pour les articles d'une commande, je garde le nombre d'articles, le prix total et les frais de port totaux. J'obtiens alors une seule ligne par commande.

In [ ]:
articles_par_commande = items.groupby("order_id").agg(
    nb_articles=("order_item_id", "count"),       # combien d'articles
    nb_produits=("product_id", "nunique"),        # combien de produits différents
    nb_vendeurs=("seller_id", "nunique"),         # combien de vendeurs différents
    montant_produits=("price", "sum"),            # prix total
    frais_port=("freight_value", "sum"),          # frais de port totaux
).reset_index()
articles_par_commande.head()

In [ ]:
paiements_par_commande = payments.groupby("order_id").agg(
    montant_paye=("payment_value", "sum"),                    # total payé
    nb_paiements=("payment_sequential", "count"),             # en combien de paiements
    type_paiement=("payment_type", lambda s: s.mode()[0]),    # le moyen de paiement le plus utilisé
    nb_echeances_max=("payment_installments", "max"),         # payé en combien de fois au maximum
).reset_index()
paiements_par_commande.head()

Pour les avis, quand il y en a plusieurs, je garde la note du **plus récent** (je trie par date, puis je prends le dernier avec `last`).

In [ ]:
avis_par_commande = (reviews_propre.sort_values("review_answer_timestamp")
                     .groupby("order_id").agg(
                         note=("review_score", "last"),
                         nb_avis=("review_id", "count"),
                         a_laisse_un_commentaire=("review_comment_message", lambda s: s.notna().any()),
                     ).reset_index())
avis_par_commande.head()

Pour la géolocalisation, je garde un seul point par code postal : la **médiane** des coordonnées (moins sensible aux valeurs extrêmes que la moyenne) et la ville la plus fréquente.

In [ ]:
geo_par_code_postal = geo_propre.groupby("geolocation_zip_code_prefix").agg(
    lat=("geolocation_lat", "median"),
    lng=("geolocation_lng", "median"),
    ville=("geolocation_city", lambda s: s.mode()[0]),
    etat=("geolocation_state", lambda s: s.mode()[0]),
).reset_index()
geo_par_code_postal.head()

# 5. Table finale : une ligne par commande

Maintenant que tout est à une ligne par commande, je peux fusionner les tables avec `merge()`. J'utilise `how="left"` pour garder toutes les commandes, même celles qui n'ont pas d'avis ou de paiement.

In [ ]:
commandes = (orders_propre
    .merge(customers, on="customer_id", how="left")
    .merge(articles_par_commande, on="order_id", how="left")
    .merge(paiements_par_commande, on="order_id", how="left")
    .merge(avis_par_commande, on="order_id", how="left"))

# Je vérifie qu'il n'y a pas de commande en double après les fusions
assert commandes.order_id.is_unique, "Attention : des commandes sont en double !"
print(commandes.shape)
commandes.head()